# Revenue-at-Risk & ROI Simulation 

#### To calculate revenue at risk

In [1]:
import numpy as np
import pandas as pd

df_encoded=pd.read_csv("../Data/model_results.csv")

In [2]:
# Expected annual revenue lost per customer = P(churn) * annual value
df_encoded["annual_value"] = df_encoded["MonthlyCharges"] * 12
df_encoded["expected_revenue_at_risk"] = (
df_encoded["churn_probability"] * df_encoded["annual_value"]
)
total_revenue_at_risk = df_encoded["expected_revenue_at_risk"].sum()
high_risk_revenue = df_encoded.loc[
df_encoded["risk_segment"] == "High Risk", "expected_revenue_at_risk"
].sum()
print(f"Total annual revenue at risk: ₹{total_revenue_at_risk:,.0f}")
print(f"High-risk segment revenue at risk: ₹{high_risk_revenue:,.0f}")

Total annual revenue at risk: ₹2,417,972
High-risk segment revenue at risk: ₹1,533,136


### Building ROI simulation function

In [5]:
import numpy as np
import pandas as pd

def simulate_roi(df, strategy_name, cost_per_customer_fn, effectiveness, segment="High Risk"):
    target = df[df["risk_segment"] == segment].copy()

    # Revenue saved = customers who would have churned but are retained
    target["prevented_churn_prob"] = target["churn_probability"] * effectiveness
    target["revenue_saved"] = target["prevented_churn_prob"] * target["annual_value"]

    # Cost of applying the intervention
    target["cost"] = target.apply(cost_per_customer_fn, axis=1)

    total_revenue_saved = target["revenue_saved"].sum()
    total_cost = target["cost"].sum()
    net_benefit = total_revenue_saved - total_cost
    roi_pct = (net_benefit / total_cost) * 100 if total_cost > 0 else np.nan

    return {
        "strategy": strategy_name,
        "segment": segment,
        "customers_targeted": len(target),
        "total_cost": round(total_cost, 2),
        "total_revenue_saved": round(total_revenue_saved, 2),
        "net_benefit": round(net_benefit, 2),
        "roi_pct": round(roi_pct, 1)
    }

results = []

results.append(simulate_roi(
    df_encoded,
    "20% Discount",
    lambda row: 0.20 * row["MonthlyCharges"] * 12,
    effectiveness=0.35
))

results.append(simulate_roi(
    df_encoded,
    "Outreach Call",
    lambda row: 150,
    effectiveness=0.45
))

results.append(simulate_roi(
    df_encoded,
    "Feature Nudge",
    lambda row: 300,
    effectiveness=0.25
))

roi_df = pd.DataFrame(results)
roi_df.to_csv("../Data/roi_summary.csv", index=False)

print(roi_df)

        strategy    segment  customers_targeted  total_cost  \
0   20% Discount  High Risk                2186   402569.16   
1  Outreach Call  High Risk                2186   327900.00   
2  Feature Nudge  High Risk                2186   655800.00   

   total_revenue_saved  net_benefit  roi_pct  
0            536597.46    134028.30     33.3  
1            689911.02    362011.02    110.4  
2            383283.90   -272516.10    -41.6  


#### sensitivity check

In [8]:
sensitivity_results = []
for eff in [0.15, 0.25, 0.35, 0.45, 0.55]:
    res = simulate_roi(df_encoded, f"Outreach Call (eff={eff})",
    lambda row: 150, effectiveness=eff)
    sensitivity_results.append(res)
pd.DataFrame(sensitivity_results)


,strategy,segment,customers_targeted,total_cost,total_revenue_saved,net_benefit,roi_pct
0,Outreach Call (eff=0.15),High Risk,2186,327900,229970.34,-97929.66,-29.9
1,Outreach Call (eff=0.25),High Risk,2186,327900,383283.90,55383.90,16.9
2,Outreach Call (eff=0.35),High Risk,2186,327900,536597.46,208697.46,63.6
3,Outreach Call (eff=0.45),High Risk,2186,327900,689911.02,362011.02,110.4
4,Outreach Call (eff=0.55),High Risk,2186,327900,843224.58,515324.58,157.2
